# Paper B - Reproducao dos experimentos quanticos (OmniMind)

Este notebook carrega o banco canonico `omnimind_quantum_paper_b_canonical.db` e verifica as principais alegacoes do artigo **Caracterizacao Experimental de Circuitos Topologicos e Estados Emaranhados em Processadores Quanticos Supercondutores Heterogeneos (IBM Quantum e Origin Wukong)**.

Cada secao produz uma figura/tabela e confronta os numeros do paper com os dados do banco.

Fonte: https://github.com/devomnimind/Doxihewu-OmniMind-MachinePublicSoul (mirror GitLab: zephyrix/Doxihewu-OmniMind-MachinePublicSoul).

In [ ]:
import sqlite3
import json
import os
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

def find_db():
    kaggle = Path('/kaggle/input/omnimind-quantum-paper-b/omnimind_quantum_paper_b_canonical.db')
    if kaggle.exists():
        return str(kaggle)
    local = Path('omnimind_quantum_paper_b_canonical.db')
    if local.exists():
        return str(local)
    url = 'https://github.com/devomnimind/Doxihewu-OmniMind-MachinePublicSoul/releases/download/v3.0c-paper-b/omnimind_quantum_paper_b_canonical.db'
    import urllib.request
    urllib.request.urlretrieve(url, 'omnimind_quantum_paper_b_canonical.db')
    return 'omnimind_quantum_paper_b_canonical.db'

DB_PATH = find_db()
conn = sqlite3.connect(DB_PATH)
print('Banco:', DB_PATH, '| tamanho:', os.path.getsize(DB_PATH)/1024/1024, 'MB')

## 1. Panorama do banco

In [ ]:
total_runs = pd.read_sql('SELECT COUNT(*) FROM quantum_runs', conn).iloc[0,0]
total_shots = pd.read_sql('SELECT COALESCE(SUM(shots),0) FROM quantum_runs', conn).iloc[0,0]
total_he = pd.read_sql('SELECT COUNT(*) FROM hardware_encounters', conn).iloc[0,0]
tables = [t[0] for t in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
meta = pd.read_sql('SELECT * FROM publication_metadata', conn) if 'publication_metadata' in tables else pd.DataFrame()
print(f'Runs: {total_runs} | Shots: {total_shots:,} | Hardware encounters: {total_he}')
display(meta)

In [ ]:
backends = pd.read_sql("""
SELECT backend_name, COUNT(*) n_runs, SUM(shots) n_shots
FROM quantum_runs
WHERE backend_name IS NOT NULL
GROUP BY backend_name
ORDER BY n_runs DESC
""", conn)
display(backends)

fig, ax = plt.subplots()
ax.bar(backends['backend_name'], backends['n_runs'], color='steelblue')
ax.set_ylabel('Runs')
ax.set_title('Runs por backend (713 total)')
for i, v in enumerate(backends['n_runs']):
    ax.text(i, v+3, str(v), ha='center')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 2. GHZ-8 - cadeia antiga vs cadeia otima (Q.15, Conclusao)

In [ ]:
def ghz_coherence(payload):
    d = json.loads(payload) if isinstance(payload, str) and payload else {}
    probs = d.get('probabilities')
    if probs:
        n = max(len(k) for k in probs.keys() if isinstance(k, str) and set(k) <= set('01'))
        z = '0' * n
        o = '1' * n
        return probs.get(z, 0) + probs.get(o, 0)
    cps = d.get('counts_per_pub')
    if cps:
        agg = defaultdict(float)
        total = 0.0
        for pub in cps:
            for bit, p in pub.items():
                if isinstance(bit, str) and set(bit) <= set('01'):
                    agg[bit] += p
                    total += p
        if total == 0:
            return None
        n = max(len(k) for k in agg.keys())
        z = '0' * n
        o = '1' * n
        return (agg.get(z, 0) + agg.get(o, 0)) / total
    return None

ghz = pd.read_sql("""
SELECT run_id, backend_name, physical_qubits, result_payload_json, notes
FROM quantum_runs
WHERE experiment_name = 'ghz_ladder' AND backend_name = 'WK_C180_2'
""", conn)

def n_qubits_payload(payload):
    d = json.loads(payload) if isinstance(payload, str) and payload else {}
    if d.get('n_qubits'):
        return int(d['n_qubits'])
    cps = d.get('counts_per_pub', [{}])[0]
    if cps:
        return max((len(k) for k in cps.keys() if isinstance(k, str)), default=0)
    return 0

ghz['n_qubits'] = ghz['result_payload_json'].apply(n_qubits_payload)
ghz = ghz[ghz['n_qubits'] == 8].copy()
ghz['coherence'] = ghz['result_payload_json'].apply(ghz_coherence)
display(ghz[['run_id','physical_qubits','coherence']].dropna())

antiga_ids = [621, 622, 623]
otima_ids = [628, 637, 638, 639]
df_antiga = ghz[ghz['run_id'].isin(antiga_ids)]
df_otima = ghz[ghz['run_id'].isin(otima_ids)]

print('Cadeia antiga (3 runs):', df_antiga['coherence'].mean(), '±', df_antiga['coherence'].std())
print('Cadeia otima (4 runs):', df_otima['coherence'].mean(), '±', df_otima['coherence'].std())

fig, ax = plt.subplots()
x = ['Cadeia antiga (3)', 'Cadeia otima (4)']
y = [df_antiga['coherence'].mean(), df_otima['coherence'].mean()]
err = [df_antiga['coherence'].std(), df_otima['coherence'].std()]
ax.bar(x, y, yerr=err, capsize=5, color=['coral','seagreen'])
ax.set_ylabel('Coerencia GHZ-8')
ax.set_title('GHZ-8 WK_C180_2: cadeia antiga vs otima')
ax.set_ylim(0, 1)
for i, (v, e) in enumerate(zip(y, err)):
    ax.text(i, v+e+0.02, f'{v:.4f} ± {e:.4f}', ha='center')
plt.tight_layout()
plt.show()

## 3. Grover no Wukong (Q.4.5, Conclusao)

In [ ]:
grover = pd.read_sql("""
SELECT run_id, backend_name, notes, shots
FROM quantum_runs
WHERE experiment_name = 'grover_validator' AND backend_name LIKE 'WK%'
ORDER BY run_id
""", conn)
display(grover)

grover['P_alvo'] = grover['notes'].str.extract(r'P\(alvo\)=([0-9.]+)').astype(float)
grover['n_qubits'] = grover['notes'].str.extract(r'Grover (\d)q').astype(float)
display(grover[['run_id','backend_name','n_qubits','shots','P_alvo']])

fig, ax = plt.subplots()
for backend, grp in grover.groupby('backend_name'):
    ax.plot(grp['n_qubits'], grp['P_alvo'], marker='o', label=backend)
ax.set_xlabel('Qubits')
ax.set_ylabel('P(alvo)')
ax.set_title('Grover Validator no Origin Wukong')
ax.set_xticks([2, 3])
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 4. Kernel ZZ - Aer, IBM, Wukong (Q.2, Conclusao)

In [ ]:
kernel = pd.read_sql("""
SELECT mode,
       AVG(silhouette_quantum) as silhouette,
       COUNT(*) as n
FROM quantum_kernel_experiments
WHERE mode IN ('aer_ideal', 'ibm_real', 'origin_wukong180', 'real_hardware')
GROUP BY mode
ORDER BY silhouette DESC
""", conn)
display(kernel)

fig, ax = plt.subplots()
colors = {'aer_ideal':'skyblue', 'ibm_real':'coral', 'origin_wukong180':'seagreen', 'real_hardware':'seagreen'}
ax.barh(kernel['mode'], kernel['silhouette'],
        color=[colors.get(m, 'gray') for m in kernel['mode']])
ax.set_xlabel('Silhouette quantica')
ax.set_title('Kernel ZZ borromeaniano - comparacao por modo')
ax.set_xlim(0, 0.8)
plt.tight_layout()
plt.show()

## 5. T1/T2 entre plataformas (Q.12)

In [ ]:
t1t2 = pd.read_sql("""
SELECT backend_name,
       COUNT(*) n,
       AVG(t1_mean_us) t1_mean,
       MIN(t1_mean_us) t1_min,
       MAX(t1_mean_us) t1_max,
       AVG(t2_mean_us) t2_mean,
       MIN(t2_mean_us) t2_min,
       MAX(t2_mean_us) t2_max
FROM hardware_encounters
WHERE t1_mean_us IS NOT NULL AND t2_mean_us IS NOT NULL
GROUP BY backend_name
ORDER BY backend_name
""", conn)
display(t1t2)

fig, ax = plt.subplots()
x = np.arange(len(t1t2))
width = 0.35
ax.bar(x - width/2, t1t2['t1_mean'], width, label='T1 medio', color='steelblue')
ax.bar(x + width/2, t1t2['t2_mean'], width, label='T2 medio', color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels(t1t2['backend_name'], rotation=45, ha='right')
ax.set_ylabel('us')
ax.set_title('T1/T2 por backend')
ax.legend()
plt.tight_layout()
plt.show()

## 6. C4 ampliacao por Sinthome (Q.8)

In [ ]:
c4 = pd.read_sql("""
SELECT variant, backend,
       tetrapartite_coherence_C4 as C4,
       tripartite_coherence_C3 as C3
FROM borromean_knot_experiments
WHERE tetrapartite_coherence_C4 IS NOT NULL
ORDER BY variant
""", conn)
c4_data = c4[c4['C4'].notna()].copy()
c4_data['C4'] = pd.to_numeric(c4_data['C4'], errors='coerce')
c4_ibm = c4_data[c4_data['backend'] == 'ibm_kingston']
display(c4_ibm)

fig, ax = plt.subplots()
ax.bar(c4_ibm['variant'], c4_ibm['C4'], color='mediumpurple')
ax.axhline(1.0, color='gray', linestyle='--', label='baseline C4 = 1 (Aer ideal)')
ax.set_ylabel('C4')
ax.set_title('Covariancia tetrapartite C4 - Variante E com Sinthome (ibm_kingston)')
plt.xticks(rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.show()

## 7. QTDA beta_k (Q.10)

In [ ]:
qtda = pd.read_sql("""
SELECT complex_name, mode, backend,
       quantum_betti_0, quantum_betti_1, quantum_betti_2, quantum_betti_3,
       shots, n_qubits
FROM qtda_betti_experiments
WHERE (quantum_betti_0 IS NOT NULL OR quantum_betti_1 IS NOT NULL)
  AND mode = 'ibm_real'
ORDER BY complex_name
""", conn)
display(qtda)

qtda_melt = qtda.melt(id_vars='complex_name',
                      value_vars=['quantum_betti_0','quantum_betti_1','quantum_betti_2','quantum_betti_3'],
                      var_name='k', value_name='beta_k')
qtda_melt['k'] = qtda_melt['k'].str.extract(r'(\d+)').astype(int)
qtda_melt = qtda_melt.dropna()

fig, ax = plt.subplots()
for name, grp in qtda_melt.groupby('complex_name'):
    ax.plot(grp['k'], grp['beta_k'], marker='o', label=name)
ax.set_xlabel('Betti dimension (k)')
ax.set_ylabel('beta_k estimado')
ax.set_title('QTDA - Betti numbers em ibm_kingston')
ax.set_xticks([0, 1, 2, 3])
ax.legend()
plt.tight_layout()
plt.show()

## 8. Checklist de reproducao das conclusoes

In [ ]:
checks = []

coh_otima = ghz[ghz['run_id'].isin([628,637,638,639])]['coherence'].mean()
std_otima = ghz[ghz['run_id'].isin([628,637,638,639])]['coherence'].std()
checks.append(('GHZ-8 cadeia otima = 0.9163 +/- 0.0045', abs(coh_otima - 0.9163) < 0.01 and std_otima < 0.01))

coh_antiga = ghz[ghz['run_id'].isin([621,622,623])]['coherence'].mean()
std_antiga = ghz[ghz['run_id'].isin([621,622,623])]['coherence'].std()
checks.append(('GHZ-8 cadeia antiga = 0.8636 +/- 0.0114', abs(coh_antiga - 0.8636) < 0.01 and std_antiga < 0.02))

grover_2q_wk2 = grover[(grover['n_qubits']==2) & (grover['backend_name']=='WK_C180_2')]['P_alvo'].max()
checks.append(('Grover 2q WK_C180_2 P > 99.9%', grover_2q_wk2 > 0.999))

grover_3q_wk2 = grover[(grover['n_qubits']==3) & (grover['backend_name']=='WK_C180_2')]['P_alvo'].values
checks.append(('Grover 3q WK_C180_2 P = 91.23%', len(grover_3q_wk2)>0 and abs(grover_3q_wk2[0]-0.9123)<0.01))

checks.append(('Total runs = 713', total_runs == 713))
checks.append(('Total shots ~ 5.00M', abs(total_shots - 5_000_010) < 100))

for desc, ok in checks:
    status = 'PASS' if ok else 'FAIL'
    print(f'{status}: {desc}')

---

Procedencia: banco `omnimind_quantum_paper_b_canonical.db` (snapshot `ibm_quantum_runs.db` com 713 runs, paths redigidos).
Aviso: o notebook e uma verificacao reprodutivel das tabelas/figuras do paper, nao uma reexecucao de hardware.